# SO CBSO — Central Bank of Somalia

Jira: DECD-6293. Five licensed-entity lists on centralbank.gov.so, all the same Elementor block-grid layout.

**Note:** entity names are NOT published as text on these pages (each block is a logo image with empty `alt` + address/website/email/phone). `Name` is derived from the logo filename, falling back to the website domain, then the email domain — flag for human validation.

In [1]:
import os
import re
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import urllib3
urllib3.disable_warnings()

In [2]:
regulatorName = 'SO CBSO'

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])
processdate = now.strftime('%Y-%m-%d')

# ------ At first we will define the workspace path -----
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)
print(scriptfolder)
print(filename)

/Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/SO CBSO
SO CBSO SQL Ready 2026-07-20 18.14.13.xlsx


In [3]:
# Lists from Jira DECD-6293. ListLabel: 1 = bank lists, 2 = insurance, 4 = everything else.
regdict = {
    1: {"ListName": "Licensed Banks",
        "URL": "https://centralbank.gov.so/licensed-banks/",
        "ListLabel": 1,
        "Comments": "Each block is an entity, extract the info from all of them."},
    2: {"ListName": "Licensed Money Transfer Businesses",
        "URL": "https://centralbank.gov.so/licensed-money-transfer/",
        "ListLabel": 4,
        "Comments": "Each block is an entity, extract the info from all of them."},
    3: {"ListName": "Licensed Mobile Money Service Providers",
        "URL": "https://centralbank.gov.so/mobile-money-service-providers/",
        "ListLabel": 4,
        "Comments": "Each block is an entity, extract the info from all of them."},
    4: {"ListName": "Licensed Microfinance Institutions",
        "URL": "https://centralbank.gov.so/licensed-microfinance-institutions/",
        "ListLabel": 4,
        "Comments": "NEW LIST! Each block is an entity, extract the info from all of them."},
    5: {"ListName": "Licensed Takaful Operators",
        "URL": "https://centralbank.gov.so/licensed-takaful-operators/",
        "ListLabel": 2,
        "Comments": "NEW LIST! Each block is an entity, extract the info from all of them."},
}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

In [4]:
def bourange_same_length_array(sqldict):
    ## pad every column with '' up to the number of processed rows
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        while len(sqldict[key]) < maxlen:
            sqldict[key].append('')
    return sqldict

In [5]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

def get_soup(url):
    r = requests.get(url, headers=HEADERS, verify=False, timeout=60)
    r.raise_for_status()
    return BeautifulSoup(r.text, 'html.parser')

def clean(text):
    return re.sub(r'\s+', ' ', text.replace('\xa0', ' ')).strip()

## filename tokens that mean "this is not a name" (photo dumps used as logos)
JUNKWORDS = ('whatsapp', 'image', 'img', 'photo', 'screenshot')

def name_from_logo(src):
    ## logo filename like 'Amana-Express-e1734371872302.webp' -> 'Amana Express';
    ## returns '' when the filename carries no usable name so the caller falls back
    base = os.path.basename(urlparse(src).path)
    base = re.sub(r'(\.(webp|png|jpe?g|gif|svg))+$', '', base, flags=re.I)
    base = re.sub(r'-e\d{10,}$', '', base)          ## WordPress edit-timestamp suffix
    base = re.sub(r'-\d+x\d+$', '', base)
    base = re.sub(r'[-_]scaled$', '', base, flags=re.I)
    tokens = [re.sub(r'(logo|lgo)$', '', w, flags=re.I) for w in re.sub(r'[-_]+', ' ', base).split()]
    tokens = [w for w in tokens if w and re.search(r'[A-Za-z]', w) and not re.fullmatch(r'\d+x\d+', w)]
    ## words that are file/asset noise, never part of a company name
    NOISE = ('png', 'jpg', 'jpeg', 'webp', 'svg', 'scaled', 'cropped', 'official', 'final', 'page', 'new')
    tokens = [w for w in tokens if w.lower() not in NOISE]
    if any(w.lower() in JUNKWORDS for w in tokens):
        return ''
    words = clean(' '.join(tokens))
    if not words or re.search(r'\d{4,}', words) or len(re.sub(r'[^A-Za-z]', '', words)) < 3:
        return ''
    return ' '.join(w if w.isupper() else w.capitalize() for w in words.split())

def name_from_domain(url):
    d = re.sub(r'^https?://', '', url.strip().lower())
    d = re.sub(r'^w{3,}\.', '', d).split('/')[0]
    core = d.split('.')[0]
    if not core or not re.search(r'[a-z]', core):
        return ''
    return ' '.join(w.capitalize() for w in core.split('-') if w)

CITY_RE = re.compile(r',\s*([A-Za-z\' ]+?)\s*,?\s*Somalia\s*$', re.I)

def parse_city(address):
    m = CITY_RE.search(address)
    return clean(m.group(1)) if m else ''

In [6]:
def parse_entities(soup):
    ## each entity = one direct-child cell of the single Elementor grid;
    ## CBS's own contact block sits outside the grid so it is excluded automatically
    grid = soup.select_one('div.e-grid.e-con')
    entities = []
    for cell in grid.find_all('div', class_='e-con', recursive=False):
        ent = {'logo': '', 'address': '', 'website': '', 'email': '', 'phone': ''}
        img = cell.select_one('div[data-widget_type="image.default"] img')
        if img:
            ent['logo'] = img.get('src', '')
        for li in cell.select('ul.elementor-icon-list-items > li'):
            icon = li.select_one('.elementor-icon-list-icon svg')
            iconclass = ' '.join(icon.get('class', [])) if icon else ''
            span = li.select_one('span.elementor-icon-list-text')
            if span is None:
                continue
            if 'location' in iconclass:
                ent['address'] = clean(span.get_text(' '))
            elif 'globe' in iconclass:
                ## one span holds website and/or email, separated by <br> or a plain
                ## space, sometimes a literal 'N/A' -- pull both out by regex
                gtext = clean(span.get_text(' '))
                m = re.search(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}', gtext)
                if m:
                    ent['email'] = m.group(0)
                    gtext = gtext.replace(m.group(0), ' ')
                w = re.search(r'(?:https?://)?(?:w{3,}\.)?[\w-]+(?:\.[\w-]+)+(?:/\S*)?', gtext)
                if w:
                    ent['website'] = w.group(0)
            elif 'phone' in iconclass:
                ent['phone'] = clean(span.get_text(' '))
        entities.append(ent)
    return entities

In [7]:
for nr, info in regdict.items():
    print('List {}: {}'.format(nr, info['ListName']))
    soup = get_soup(info['URL'])
    entities = parse_entities(soup)
    print('  {} entities'.format(len(entities)))
    for ent in entities:
        name = name_from_logo(ent['logo'])
        if not name or re.search(r'\d', name):
            ## digits in a filename-derived name = low confidence; prefer domain/email
            alt = name_from_domain(ent['website'])
            if not alt and ent['email']:
                local, _, dom = ent['email'].partition('@')
                core = dom.split('.')[0].lower()
                ## generic mail providers carry no entity name -> use the mailbox name
                alt = local.capitalize() if core in ('gmail', 'hotmail', 'yahoo', 'outlook', 'live') else ' '.join(w.capitalize() for w in core.split('-') if w)
            name = alt or name
        sqldict['Name'].append(name)
        sqldict['Address_1'].append(ent['address'])
        sqldict['City'].append(parse_city(ent['address']))
        sqldict['Cntry'].append('SO')
        sqldict['Phone'].append(ent['phone'])
        sqldict['Website'].append(ent['website'])
        sqldict['Email'].append(ent['email'])
        sqldict['ListLabel'].append(info['ListLabel'])
        sqldict['ListName'].append(info['ListName'])
        sqldict['ListCode'].append(str(nr))
        sqldict['ListLanguage'].append('EN')
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegCtry'].append('SO')
        sqldict['RegCode'].append('CBSO')
        sqldict['ListProcessDate'].append(processdate)
        sqldict = bourange_same_length_array(sqldict)

List 1: Licensed Banks


  15 entities
List 2: Licensed Money Transfer Businesses


  17 entities
List 3: Licensed Mobile Money Service Providers


  6 entities
List 4: Licensed Microfinance Institutions


  20 entities
List 5: Licensed Takaful Operators


  7 entities


In [8]:
# ------ Final step we will save the df to .xlsx file ----
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 65 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/SO CBSO/SO CBSO SQL Ready 2026-07-20 18.14.13.xlsx


In [9]:
## quick QA: per-list counts and required-field coverage
print('Total rows:', len(df))
print()
print(df.groupby(['ListCode', 'ListName']).size().to_string())
print()
for col in ['Name', 'Address_1', 'City', 'Phone', 'Website', 'Email']:
    print('{:<10} non-empty: {}/{}'.format(col, (df[col] != '').sum(), len(df)))

Total rows: 65

ListCode  ListName                               
1         Licensed Banks                             15
2         Licensed Money Transfer Businesses         17
3         Licensed Mobile Money Service Providers     6
4         Licensed Microfinance Institutions         20
5         Licensed Takaful Operators                  7

Name       non-empty: 65/65
Address_1  non-empty: 65/65
City       non-empty: 34/65
Phone      non-empty: 65/65
Website    non-empty: 57/65
Email      non-empty: 62/65
